##### 1. Import libraries & Load Datasets

In [17]:
import pandas as pd
import sqlite3

# Load dataset
df = pd.read_csv("../data/Electric_Vehicle_Population_Data.csv")

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279780 entries, 0 to 279779
Data columns (total 16 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   VIN (1-10)                                         279780 non-null  object 
 1   County                                             279756 non-null  object 
 2   City                                               279756 non-null  object 
 3   State                                              279780 non-null  object 
 4   Postal Code                                        279756 non-null  float64
 5   Model Year                                         279780 non-null  int64  
 6   Make                                               279780 non-null  object 
 7   Model                                              279780 non-null  object 
 8   Electric Vehicle Type                              279780 non-null  object

In [19]:
df.describe()

,Postal Code,Model Year,Electric Range,Legislative District,DOL Vehicle ID,2020 Census Tract
count,279756.000000,279780.000000,279769.000000,279080.000000,2.797800e+05,2.797560e+05
mean,98176.111447,2022.074072,39.172256,28.827841,2.462994e+08,5.297189e+10
std,2577.398040,3.059852,78.230356,14.908133,6.346205e+07,1.636194e+09
min,1030.000000,1999.000000,0.000000,1.000000,4.385000e+03,1.001020e+09
25%,98052.000000,2021.000000,0.000000,17.000000,2.213867e+08,5.303301e+10
50%,98133.000000,2023.000000,0.000000,32.000000,2.629453e+08,5.303303e+10
75%,98382.000000,2024.000000,32.000000,42.000000,2.790832e+08,5.305394e+10
max,99517.000000,2027.000000,337.000000,49.000000,4.791150e+08,6.601095e+10


In [20]:
df.head(3)

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
0,JN1AZ0CP5C,Stevens,Colville,WA,99114.0,2012,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73.0,7.0,153331706,POINT (-117.90454 48.54657),AVISTA CORP,5.306595e+10
1,JTMABABA7P,Yakima,Yakima,WA,98903.0,2023,SUBARU,SOLTERRA,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,15.0,253586308,POINT (-120.71847 46.55029),PACIFICORP,5.307700e+10
2,1N4AZ1CP1J,King,Seattle,WA,98122.0,2018,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,151.0,37.0,333135022,POINT (-122.31009 47.60803),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10


In [21]:
# null values
df.isnull().sum()

VIN (1-10)                                             0
County                                                24
City                                                  24
State                                                  0
Postal Code                                           24
Model Year                                             0
Make                                                   0
Model                                                  0
Electric Vehicle Type                                  0
Clean Alternative Fuel Vehicle (CAFV) Eligibility      0
Electric Range                                        11
Legislative District                                 700
DOL Vehicle ID                                         0
Vehicle Location                                     109
Electric Utility                                      24
2020 Census Tract                                     24
dtype: int64

##### Convert model year

In [22]:
df["Model Year"] = pd.to_numeric(df["Model Year"], errors="coerce")

##### 2. Remove duplicates

In [23]:
df = df.drop_duplicates()

##### 4. Standardize column names

In [25]:
df.columns = df.columns.str.replace(" ", "_")

print(df.head())

   VIN_(1-10)    County      City State  Postal_Code  Model_Year    Make  \
0  JN1AZ0CP5C   Stevens  Colville    WA      99114.0        2012  NISSAN   
1  JTMABABA7P    Yakima    Yakima    WA      98903.0        2023  SUBARU   
2  1N4AZ1CP1J      King   Seattle    WA      98122.0        2018  NISSAN   
3  5UX43EU09S    Kitsap   Poulsbo    WA      98370.0        2025     BMW   
4  3C3CFFGE5F  Thurston      Yelm    WA      98597.0        2015    FIAT   

      Model                   Electric_Vehicle_Type  \
0      LEAF          Battery Electric Vehicle (BEV)   
1  SOLTERRA          Battery Electric Vehicle (BEV)   
2      LEAF          Battery Electric Vehicle (BEV)   
3        X5  Plug-in Hybrid Electric Vehicle (PHEV)   
4       500          Battery Electric Vehicle (BEV)   

   Clean_Alternative_Fuel_Vehicle_(CAFV)_Eligibility  Electric_Range  \
0            Clean Alternative Fuel Vehicle Eligible            73.0   
1  Eligibility unknown as battery range has not b...             0.0

##### 5. Remove complex outliers

In [26]:
# Detect numeric columns
numeric_columns = df.select_dtypes(include=["int64","float64"]).columns

# Remove identifier columns because they are not relevant for outlier detection
exclude_columns = ["Postal_Code","Legislative_District","2020_Census_Tract"]

numeric_columns = [col for col in numeric_columns if col not in exclude_columns]

# Outlier removal using IQR
for col in numeric_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

print("Outliers removed")
print(df.info())


Outliers removed
<class 'pandas.core.frame.DataFrame'>
Index: 211963 entries, 1 to 279779
Data columns (total 16 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   VIN_(1-10)                                         211963 non-null  object 
 1   County                                             211943 non-null  object 
 2   City                                               211943 non-null  object 
 3   State                                              211963 non-null  object 
 4   Postal_Code                                        211943 non-null  float64
 5   Model_Year                                         211963 non-null  int64  
 6   Make                                               211963 non-null  object 
 7   Model                                              211963 non-null  object 
 8   Electric_Vehicle_Type                              211963 non-

##### 6. Handle missing values in columns

In [28]:
df["Electric_Range"] = df["Electric_Range"].replace(0, pd.NA)
for col in numeric_columns:

    mean_value = df[col].mean()
    median_value = df[col].median()
    mode_value = df[col].mode()[0]

    print(f"{col}")
    print(f"Mean   : {mean_value}")
    print(f"Median : {median_value}")
    print(f"Mode   : {mode_value}")
    print("---------------------")

Model_Year
Mean   : 2023.4380858923491
Median : 2023.0
Mode   : 2023
---------------------
Electric_Range
Mean   : 31.623299134104634
Median : 32.0
Mode   : 32.0
---------------------
DOL_Vehicle_ID
Mean   : 258604994.17425683
Median : 268106226.0
Mode   : 172161804
---------------------


###### Exploratory data analysis showed that most vehicles in the dataset are recent models, with 2023 being the dominant model year. Additionally, the typical electric driving range is approximately 32 miles, indicating a large proportion of plug-in hybrid electric vehicles in the dataset.

##### 6. Save Statistics to CSV

In [29]:
stats = []

for col in numeric_columns:

    stats.append({
        "column": col,
        "mean": df[col].mean(),
        "median": df[col].median(),
        "mode": df[col].mode()[0]
    })

stats_df = pd.DataFrame(stats)

stats_df.to_csv("../data/numeric_statistics.csv", index=False)

##### 7. Save Cleaned model

In [31]:
df.to_csv("../data/ev_cleaned.csv", index=False)

##### 8. Create SQL Database

In [32]:
conn = sqlite3.connect("../database/ev_database.db")

df.to_sql(
    "ev_data",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("EV dataset loaded to SQL database")

EV dataset loaded to SQL database
